* `nse.py` programs   
* [x] Get NSE symbols   
* [x] Build option chains   
* [x] Get accurate dte
* [ ] Get prices   
* [ ] Get margins   



## Get nse symbols

In [ ]:
# hack to change jupyter directory in notebooks for imports
import os
from pathlib import Path
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
else:
    root = Path.cwd()
os.chdir(root)

# Logger
import logging, logging.config
import yaml

with open(root / 'config' / 'log.yml', 'r') as f:
    config = yaml.safe_load(f.read())
    logging.config.dictConfig(config)

log = logging.getLogger('ib_log')

In [ ]:
# get the symbols
from src.nse.nse import get_nse_syms
df_syms = get_nse_syms()
df_syms.head(5)

# Build Option Chains

In [ ]:
# Gets new chains if they are old

from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm_notebook

from src.nse.nse import get_nse_chain
from src.support import pickle_age

days_old = 0.75 # set how many days old
chains_file = root / 'data' / 'master'

try:
    chain_age_delta = pickle_age(chains_file)['nse_chains.pkl']
    chain_age = chain_age_delta.days + chain_age_delta.hours/24 + chain_age_delta.minutes/24/60
except KeyError:
    chain_age = days_old + 1 # force regeneration

old_chains = chain_age > days_old
dfs = []

if old_chains:
    tq_scrips = tqdm_notebook(df_syms.symbol, bar_format="{desc:<8}{percentage:3.0f}%{r_bar}")
    for s in tq_scrips:
        tq_scrips.set_description(f"{s}")
        try:
            dfs.append(get_nse_chain(s))
        except KeyError as e:
            log.error(f"{s} has error {e}")
    # dfs = [get_nse_chain(s) for s in ] # regenerate
else:
    dfs = [pd.read_pickle(chains_file / 'nse_chains.pkl')]

# assemble the chains
nse_chains = pd.concat(dfs,ignore_index=True)

# store the nse_chains if new
if old_chains:
    data_path = root / 'data' / 'master' / 'nse_chains.pkl'
    nse_chains.to_pickle(data_path)

# Get accurate dte

In [ ]:
# get the dtes for the chains
from src.support import get_dte

exchange = df_syms.exchange.iloc[0]

dte = nse_chains.expiry.apply(lambda x: get_dte(x, exchange))
dte.name = 'dte'

# insert the dtes
try:
    nse_chains.insert(5, 'dte', dte)
except ValueError:
    log.warning('dte already in nse_chains, will be refreshed')
    nse_chains = nse_chains.assign(dte=dte)

# Get prices of underlyings

In [ ]:
prices = dict()
from nsepy import get_quote

tq_scrips = tqdm_notebook(df_syms.symbol, bar_format="{desc:<8}{percentage:3.0f}%{r_bar}")
for s in tq_scrips:
    tq_scrips.set_description(f"{s}")
    try:
        price = get_quote(s)['data'][0]['lastPrice']
    except KeyError:
        price = None
    prices[s] = price

In [ ]:
get_quote('BANKNIFTY', 'EQ')

In [ ]:
import requests
import pandas as pd

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:74.0) Gecko/20100101 Firefox/74.0'
}


def main(url):
    r = requests.get(url, headers=headers).json()
    x = []
    for item in r['data']:
        df = pd.DataFrame.from_dict([item])
        x.append(df)
    new = pd.concat(x, ignore_index=True)
    return new


df = main("https://www1.nseindia.com/live_market/dynaContent/live_watch/stock_watch/foSecStockWatch.json")

In [ ]:
url = "https://www.nseindia.com/api/equity-stockIndices?index=SECURITIES%20IN%20F%26O"

s = requests.Session()
output = s.get("http://nseindia.com", headers=headers)
output = s.get(url, headers=headers).json()

In [ ]:
output['data']